# DepthWizard v5 — Kaggle 2× T4 probe: is v5 worth an H100 run?

**Accelerator `GPU T4 x2` · Internet ON · Add-ons → Secrets: `HF_TOKEN` and `GITHUB_PAT`.**
Run with **Save Version → Save & Run All (Commit)**; an interactive session idles out.

**Attach as inputs** (all `abhaydkale232/…`): `depthwizard-gamus` (train/val/test),
`depthwizard-synrs3d-g05`, `depthwizard-synrs3d-g1`, `depthwizard-india-labeled`, and
`dfc23-track2-height-estimation` (raw; packed here). v4's `best.pt` is pulled from the
private Hub repo `akashch1512/SingleViewHeightEstimator` (`best_3.7/best.pt`), so the
`HF_TOKEN` secret needs read access to it.

**Two sessions, one switch.** `WARM_START = True` starts v5 from v4's weights (the
detail branch is zero-initialised, so at step 0 v5 predicts exactly what v4 did);
`False` trains v5 from scratch. Either way it trains ~9 h with the v5 profile
(`V4_modal/v5_flags.py`, T4-sized), then scores this v5 on the held-out
`gamus/test` tiles with `eval_test.py` into `outputs/v5_probe_{v4init,scratch}/`.
Commit the notebook twice, once per setting.

**Reading it.** `test_v5_<tag>/test_metrics.json` has the plain RMSE, *edge RMSE* and
*grad ratio* (closer to 1 = sharper, not blobs). There is no v4 column: v4 scored
RMSE 10.9 on these tiles against ~3.7 on its own val, so that baseline was dropped.

**First commit: `SMOKE = True`** (~15 min) — proves clone, linking, warm start, DDP and fp16.

In [ ]:
# ═══ 1. Code, data ═══════════════════════════════════════════════════════
import os, sys, glob, shlex, subprocess, pathlib

WARM_START = True     # True: v5 starts from v4's weights.  False: from scratch.
V4_CKPT   = ""        # a Kaggle path to v4's best.pt; "" -> downloaded from V4_HF
V4_HF     = ("akashch1512/SingleViewHeightEstimator", "best_3.7/best.pt")  # private repo
SMOKE     = False     # True for the first commit
RESUME    = ""        # a dead session: "/kaggle/input/<prev>/outputs/v5_probe/last_full.pt"
MAX_MINUTES = 540     # training budget; the whole notebook is ~10.5 h of the 12 h cap
EPOCHS    = 14        # ~25 min frozen, ~45 min unfrozen on 2x T4; the clock usually ends it
DFC23_DIR = "/kaggle/input/datasets/abhaydkale232/dfc23-track2-height-estimation/track2/train"
                      # packed into dfc23_g0XX here; skipped if absent (india_labeled
                      # still exercises the coarse-label path)
SLIDING_TILES = 0     # sliding-window test tiles in cell 3; each costs ~minutes on T4
RUN_TESTS = True

REPO = "/kaggle/temp/DepthWizard"     # not /kaggle/working: the clone holds the PAT
CODE = f"{REPO}/FineTunning/v5"
DATA = "/kaggle/temp/dwdata"          # symlinks to the attached stores + the DFC23 pack
TAG  = "v4init" if WARM_START else "scratch"
OUT  = f"/kaggle/working/outputs/v5_probe_{TAG}" + ("_smoke" if SMOKE else "")
GIT_URL = "github.com/AbhayKale332/DepthWizard.git"
sh = lambda c: subprocess.run(c, shell=True, check=True)

if not os.path.isdir(REPO):
    pat = ""
    try:
        from kaggle_secrets import UserSecretsClient
        pat = UserSecretsClient().get_secret("GITHUB_PAT").strip()   # a pasted secret can carry a newline
    except Exception as e:
        print("no GITHUB_PAT secret (", e, ") - trying an anonymous clone")
    url = f"https://x-access-token:{pat}@{GIT_URL}" if pat else f"https://{GIT_URL}"
    r = subprocess.run(["git", "clone", "--depth", "1", url, REPO], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("git clone failed: " + (r.stderr.replace(pat, "***") if pat else r.stderr))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://{GIT_URL}"], check=True)
os.chdir(CODE); sys.path.insert(0, CODE)
sh("pip install -q -r requirements.txt")

if RUN_TESTS:
    r = subprocess.run("python -m pytest -q --tb=short -rf", shell=True, capture_output=True, text=True)
    print(((r.stdout or "") + (r.stderr or ""))[-6000:])
    if r.returncode:
        raise SystemExit(f"pytest exited {r.returncode} — output above. RUN_TESTS = False skips.")

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # DINOv3-SAT is gated

if not V4_CKPT and V4_HF:         # 2.5 GB; the warm-start weights
    from huggingface_hub import hf_hub_download
    V4_CKPT = hf_hub_download(*V4_HF, token=os.environ["HF_TOKEN"], local_dir="/kaggle/temp/ckpt")
if V4_CKPT and not os.path.isfile(V4_CKPT):
    raise SystemExit(f"V4_CKPT not found: {V4_CKPT}")
print(f"v4 checkpoint: {V4_CKPT or 'none'}   init: {'v4' if WARM_START else 'scratch'}")

# the attached packed stores -> one data_root of symlinks (nothing is copied)
sh(f"DW_DATA_ROOT={DATA} bash run_kaggle.sh link")
if DFC23_DIR and os.path.isdir(DFC23_DIR):
    # New Delhi (116 SuperView + 15 GF2 scenes) IS india_labeled: carve it out
    # before packing, as 01_data_prep.ipynb does, or india_labeled/val lands in
    # dfc23 train and loaders.check_split_overlaps stops the run.  The input is
    # read-only, so the rest is symlinked.
    dfc = "/kaggle/temp/dfc23_no_delhi"
    for sub in ("rgb", "dsm"):
        os.makedirs(f"{dfc}/{sub}", exist_ok=True)
        for f in glob.glob(f"{DFC23_DIR}/{sub}/*.tif"):
            name = os.path.basename(f)
            if "newdelhi" not in name.lower().replace("_", "") and not os.path.lexists(f"{dfc}/{sub}/{name}"):
                os.symlink(f, f"{dfc}/{sub}/{name}")
    n = len(glob.glob(f"{dfc}/rgb/*.tif"))
    assert n == 1642, f"1773 - 131 Delhi scenes = 1642 expected, got {n}"
    sh(f"python prepare_data.py --data_root {DATA} --datasets dfc23 --dfc23_dir {dfc} "
       f"--dfc23_tile 512 --dfc23_max_height_m 150")
sh(f"du -shL {DATA}/*; df -h /kaggle/temp | tail -1")

In [ ]:
# ═══ 1b. Continue a finished version: fetch only its last_full.pt ═════════
# Resuming a finished run with the same EPOCHS / MAX_MINUTES loads and exits at
# once.  This cell extends it: MAX_MINUTES (the total across sessions) becomes
# done + EXTRA_MINUTES and EPOCHS grows at the pace the last session ran.
# train.py holds the cosine where the last session left it rather than
# re-deriving it from the bigger budget, so a FINISHED run continues at the
# floor LR (3e-06) - fine-tuning only.  For a real second round of training,
# warm-start from that run's best.pt instead (fresh schedule from the top).
# AdamW moments, EMA, teacher, history and best all carry over.
# In a commit (non-interactive) run Kaggle only mounts a notebook's output if it
# was attached before Save Version: Add Input -> Your Work -> PREV_KERNEL's
# notebook.  It then sits under /kaggle/input, read-only, nothing copied.
# kagglehub is the fallback, and only works in an interactive session.
# Keep WARM_START as the previous version had it.
PREV_KERNEL   = ""    # e.g. "abhaydkale232/notebook8974d5468b"; "" skips this cell
EXTRA_MINUTES = 540   # this session's training time; the notebook's ~10.5 h fits

if PREV_KERNEL and not SMOKE:
    import json, math
    run = f"outputs/v5_probe_{TAG}"
    prev = "/kaggle/input"
    if not glob.glob(f"{prev}/**/{run}/last_full.pt", recursive=True):
        sh("pip install -q -U kagglehub")     # notebook_output_download is recent
        import kagglehub
        try:
            prev = kagglehub.notebook_output_download(PREV_KERNEL)
        except Exception as e:
            raise SystemExit(f"{PREV_KERNEL} is not attached ({e}).  In the editor: Add Input "
                             f"-> Your Work -> {PREV_KERNEL.split('/')[-1]}, then Save Version.")
    found = {os.path.basename(f): f for f in glob.glob(f"{prev}/**/{run}/*", recursive=True)}
    if not {"last_full.pt", "metrics.json"} <= found.keys():
        raise SystemExit(f"{PREV_KERNEL}: no {run}/last_full.pt + metrics.json in its "
                         f"output under {prev} (got {sorted(found)}) - wrong WARM_START/TAG, "
                         f"or that version ran with save_full_state off")
    RESUME = found["last_full.pt"]
    m = json.load(open(found["metrics.json"]))
    done_ep, done_min = int(m["history"][-1]["epoch"]), float(m["elapsed_min"])
    MAX_MINUTES = math.ceil(done_min + EXTRA_MINUTES)
    EPOCHS = max(done_ep + 1, math.ceil(done_ep * MAX_MINUTES / done_min))
    print(f"RESUME = {RESUME}\n  done {done_ep} epochs in {done_min:.0f} min -> "
          f"EPOCHS {EPOCHS}, MAX_MINUTES {MAX_MINUTES}; cosine continues from "
          f"where it stopped (a finished run: the floor LR)")

In [ ]:
# ═══ 2. Train: the v5 profile, T4-sized ═══════════════════════════════════
sys.path.insert(0, f"{REPO}/FineTunning/V4_modal")
from v5_flags import FLAGS as V5

# every packed train store, gamus first (best.pt is selected on the first val source)
found = sorted(pathlib.Path(p).parents[1].name for p in glob.glob(f"{DATA}/*/train/index.json"))
datasets = ",".join(["gamus"] + [n for n in found if n != "gamus"])
init = RESUME or (V4_CKPT if WARM_START else "")

flags = {**V5,
    "data_root": DATA, "output_dir": OUT, "datasets": datasets, "resume": init,
    # Turing: no bf16 -> fp16 + GradScaler; encoder checkpointing to fit 15 GB
    "amp_dtype": "fp16", "grad_checkpoint_encoder": "true",
    # 2 ranks x 1 x 16 = 32, the v5 profile's effective batch (16 x 2).  Batch 2 OOMed
    # at the e3 unfreeze (14.5 of 14.6 GiB), and fp16 now runs the decoder trunk in fp32.
    # Eval batch 16 OOMed at the first eval (fp32 trunk + upsampler at H/2); 4 leaves
    # room for the unfrozen blocks' grads and Adam state, which stay resident during eval.
    "batch_size": "1", "grad_accum": "16", "eval_batch_mult": "4",
    "num_workers": "2", "prefetch_factor": "2",
    # v5_flags unfreezes all 24 blocks at llrd 0.80; on T4 that halves throughput
    # (4.9 -> 2.3 img/s), so the probe keeps v4-T4's 16 blocks / 0.90
    "encoder_unfreeze_blocks": "16", "llrd": "0.90",
    "epochs": str(EPOCHS), "max_minutes": str(MAX_MINUTES), "session_minutes": "0",
    # held-out / sliding / TTA / ONNX at the end of train.py is hours on T4;
    # cell 3 scores it on gamus/test instead
    "test_sources": "", "final_sliding_eval": "false", "tta": "false", "export_onnx": "false",
}
if WARM_START:
    # v4's Head B range (and the resumed run's): v5_flags' 150 would stretch the
    # loaded bin widths 1.25x.  Not a weight, so a resume must pass it again.
    flags["bin_max_m"] = "120"
args = " ".join(f"--{k} {shlex.quote(str(v))}" for k, v in flags.items() if str(v) != "")
if SMOKE:
    args += " --smoke"
print("python train.py", args)

# torch 2.10 reads PYTORCH_ALLOC_CONF; the CUDA-named one is the old spelling
os.environ.update(PYTORCH_ALLOC_CONF="expandable_segments:True",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", NCCL_P2P_DISABLE="1",
                  OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")
sh(f"torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py {args}")

In [ ]:
# ═══ 3. v5 on the held-out gamus/test tiles ═══════════════════════════════
# Centre crop over all gamus/test tiles, plain (no TTA), plus 12 qualitative strips
# with shadow IoU.  No v4 baseline: it scored RMSE 10.9 here (+13.5 m tall bias,
# in bf16 too) against ~3.7 on its own val, so the comparison meant nothing.
if not SMOKE:
    common = (f"--data_root {DATA} --source gamus --split test --tiles 0 --skip_tta "
              f"--sliding_tiles {SLIDING_TILES} --qualitative 12 --max_valid_height_m 150 "
              "--batch_size 8 --eval_batch_mult 2 --num_workers 4")
    # best.pt is only written on a new best, so a resume that never beat the
    # previous session's has none of its own unless train.py carried it over
    v5_ck = next((f"{OUT}/{n}" for n in ("best.pt", "last.pt")
                  if os.path.isfile(f"{OUT}/{n}")), f"{OUT}/best.pt")
    print(f"v5 checkpoint: {v5_ck}")
    sh(f"python eval_test.py --ckpt {v5_ck} {common} --amp_dtype fp16 --out {OUT}/test_v5_{TAG}")

## If the session dies

Attach this version's output as an input to a fresh copy, set
`RESUME = "/kaggle/input/<name>/outputs/v5_probe_<v4init|scratch>/last_full.pt"`, keep
`WARM_START` as it was, and commit again. `/kaggle/temp` is wiped, so it re-clones,
re-links and re-packs DFC23 first.

## If it finished and you want more

Add Input → Your Work → that notebook, set `PREV_KERNEL` in cell 1b to its slug and
commit. It resumes from the mounted `last_full.pt` and `metrics.json`, raises `EPOCHS` /
`MAX_MINUTES` and trains `EXTRA_MINUTES` more. The LR carries on from where it stopped,
so a finished run continues at the floor LR (3e-06). Without the input
attached a commit fails: Kaggle won't attach a notebook in a non-interactive session.

Reading the scratch session: ~9 T4 hours from scratch will not reach the warm-start
session's converged numbers, so compare the two on *sharpness* (edge RMSE, grad ratio)
as well as RMSE.